# 🛡️ Agent Gateway + Agent Identity — Colab Demo

**Federated AI-agent identity on Google Cloud with Okta & Microsoft Entra ID, calling Gemini Enterprise.**

This notebook walks through the whole flow from [`avnit/demo-agent-gateway-agent-id`](https://github.com/avnit/demo-agent-gateway-agent-id) — end to end, right here in Colab.

It runs in **DEMO_MODE** by default: no GCP project, no billing, no real tokens required. Every hop is real code; only the outbound Google API calls are stubbed so `Runtime → Run all` just works. A **Live mode** section at the bottom shows exactly what to change for a real project.

### What you'll see
1. A human authenticates at **Okta** or **Entra ID** (Workforce Identity Federation).
2. The **Agent Gateway** enforces authorization policy.
3. The human's IdP token is exchanged for a **short-lived, per-agent identity** (Agent Identity).
4. The agent calls **Gemini Enterprise** *on-behalf-of* the human.
5. An **audit trail** binds the agent service account **and** the human subject to every model call.

---

## Architecture

```
Okta / Entra ID --(1) OIDC id_token--> Google STS
Google STS      --(2) federated token--> Agent Gateway
Agent Gateway   --(3) impersonate agent SA--> IAM Credentials
IAM Credentials --(4) short-lived SA token--> Agent Gateway
Agent Gateway   --(5) predict + on-behalf-of--> Gemini Enterprise
```

| # | Step | Identity primitive |
|---|------|--------------------|
| 1 | Human SSO to Okta / Entra ID | Enterprise IdP (OIDC) |
| 2 | STS token exchange | **Workforce Identity Federation** |
| 3 | Gateway selects + impersonates agent SA | **Agent Identity** |
| 4 | Short-lived token minted | IAM Credentials |
| 5 | Gemini call, human rides along | On-behalf-of + Cloud IAM |

## 0. Setup

Demo mode needs nothing but `requests`. This cell is idempotent.

In [ ]:
import os, sys, subprocess

# DEMO_MODE=1 => no live Google API calls; the federation logic still runs in full.
os.environ.setdefault('DEMO_MODE', '1')

try:
    import requests  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'requests'], check=True)

print('Setup complete. DEMO_MODE =', os.environ['DEMO_MODE'])

## 1. Enterprise IdP front doors — Okta & Entra ID

Each provider models the human completing an OIDC sign-in and returns the ID token that will be
exchanged at Google STS. In production you swap `id_token` for the real authorization-code result;
the gateway logic below is unchanged. This mirrors `app/auth/okta_provider.py` and `app/auth/azuread_provider.py`.

In [ ]:
from dataclasses import dataclass, field

@dataclass
class Identity:
    subject: str
    email: str
    groups: list
    id_token: str
    idp: str

class OktaProvider:
    idp_name = 'okta'
    def get_subject_token(self):
        return Identity(
            subject=os.environ.get('OKTA_SUBJECT', 'avnit@asbsolutions.example'),
            email=os.environ.get('OKTA_EMAIL', 'avnit@asbsolutions.example'),
            groups=['gemini-agent-users'],
            id_token=os.environ.get('OKTA_ID_TOKEN', 'DEMO.okta.oidc.id_token'),
            idp='okta',
        )

class AzureADProvider:
    idp_name = 'azuread'
    def get_subject_token(self):
        return Identity(
            subject=os.environ.get('AZUREAD_SUBJECT', 'avnit@asbsolutions.onmicrosoft.com'),
            email=os.environ.get('AZUREAD_EMAIL', 'avnit@asbsolutions.onmicrosoft.com'),
            groups=[os.environ.get('AZUREAD_GROUP_OID', 'a1b2c3d4-entra-group-oid')],
            id_token=os.environ.get('AZUREAD_ID_TOKEN', 'DEMO.entra.oidc.id_token'),
            idp='azuread',
        )

print('Providers defined: Okta, Entra ID')

## 2. Federation — STS exchange + agent-SA impersonation

The two hops that turn a human's IdP token into a **short-lived agent credential**. Mirrors `app/auth/federation.py`.
In DEMO_MODE the Google endpoints are stubbed; set `DEMO_MODE=0` (and provide ADC) to hit them for real.

In [ ]:
import json as _json

STS_ENDPOINT = 'https://sts.googleapis.com/v1/token'
IAMCRED = 'https://iamcredentials.googleapis.com/v1/projects/-/serviceAccounts/{sa}:generateAccessToken'

def _demo():
    return os.environ.get('DEMO_MODE', '1') != '0'

@dataclass
class AgentCredential:
    access_token: str
    agent_sa_email: str
    on_behalf_of_subject: str
    idp: str
    expires_in: int

def exchange_idp_token(subject_token, audience, idp):
    '''Hop 1: IdP OIDC token -> Google federated access token (STS).'''
    if _demo():
        return f'DEMO.federated.access_token.for.{idp}'
    import requests
    payload = {
        'grant_type': 'urn:ietf:params:oauth:grant-type:token-exchange',
        'audience': audience,
        'scope': 'https://www.googleapis.com/auth/cloud-platform',
        'requested_token_type': 'urn:ietf:params:oauth:token-type:access_token',
        'subject_token': subject_token,
        'subject_token_type': 'urn:ietf:params:oauth:token-type:jwt',
    }
    r = requests.post(STS_ENDPOINT, data=payload, timeout=30); r.raise_for_status()
    return r.json()['access_token']

def impersonate_agent_sa(federated_token, agent_sa_email, lifetime_seconds):
    '''Hop 2: federated token -> short-lived token for the agent SA.'''
    if _demo():
        return f'DEMO.agent_sa.access_token.{agent_sa_email}', lifetime_seconds
    import requests
    url = IAMCRED.format(sa=agent_sa_email)
    body = {'scope': ['https://www.googleapis.com/auth/cloud-platform'], 'lifetime': f'{lifetime_seconds}s'}
    headers = {'Authorization': f'Bearer {federated_token}', 'Content-Type': 'application/json'}
    r = requests.post(url, headers=headers, data=_json.dumps(body), timeout=30); r.raise_for_status()
    return r.json()['accessToken'], lifetime_seconds

def mint_agent_credential(*, subject_token, subject, audience, idp, agent_sa_email, lifetime_seconds):
    federated = exchange_idp_token(subject_token, audience, idp)
    token, ttl = impersonate_agent_sa(federated, agent_sa_email, lifetime_seconds)
    return AgentCredential(token, agent_sa_email, subject, idp, ttl)

print('Federation logic ready.')

## 3. Gemini Enterprise client

The gateway calls Gemini with the **agent's** token and labels the call with the **human** subject for audit.
Mirrors `app/gemini/client.py`.

In [ ]:
VERTEX = ('https://{region}-aiplatform.googleapis.com/v1/projects/{project}/locations/'
          '{region}/publishers/google/models/{model}:generateContent')

class GeminiEnterpriseClient:
    def __init__(self, project_id, region, model):
        self.project_id, self.region, self.model = project_id, region, model
    def generate(self, *, agent_access_token, on_behalf_of, prompt):
        if _demo():
            return (f'[DEMO Gemini Enterprise response]\n'
                    f'model={self.model} region={self.region}\n'
                    f'acting_identity=agent-sa on_behalf_of={on_behalf_of}\n'
                    f'answer: (a real call would complete: {prompt!r})')
        import requests
        url = VERTEX.format(region=self.region, project=self.project_id, model=self.model)
        headers = {'Authorization': f'Bearer {agent_access_token}', 'Content-Type': 'application/json',
                   'X-Goog-Request-Reason': f'agent-obo:{on_behalf_of}'}
        body = {'contents': [{'role': 'user', 'parts': [{'text': prompt}]}],
                'labels': {'on_behalf_of': on_behalf_of.replace('@', '_at_')}}
        r = requests.post(url, headers=headers, data=_json.dumps(body), timeout=60); r.raise_for_status()
        return r.json()['candidates'][0]['content']['parts'][0]['text']

print('Gemini Enterprise client ready.')

## 4. The Agent Gateway — broker + policy + audit

Ties it together: authenticate → authorize → federate → call → audit. Mirrors `app/agent_gateway.py`.

In [ ]:
from datetime import datetime, timezone

# --- config (mirrors app/config.py) ---
PROJECT_ID = os.environ.get('GCP_PROJECT_ID', 'my-gemini-agent-project')
REGION     = os.environ.get('GCP_REGION', 'us-central1')
WORKFORCE_POOL = os.environ.get('WORKFORCE_POOL', 'agent-gateway-workforce')
AGENT_SA   = os.environ.get('AGENT_SA_EMAIL', 'gemini-support-agent@my-gemini-agent-project.iam.gserviceaccount.com')
GEMINI_MODEL = os.environ.get('GEMINI_MODEL', 'gemini-2.5-pro')
TOKEN_TTL  = int(os.environ.get('AGENT_TOKEN_LIFETIME', '900'))
STS_AUDIENCE = f'//iam.googleapis.com/locations/global/workforcePools/{WORKFORCE_POOL}'

ALLOWED_GROUPS = {'gemini-agent-users'}

def _audit(line):
    ts = datetime.now(timezone.utc).isoformat()
    print(f'[AUDIT {ts}] {line}')

def authorize(identity):
    groups = set(identity.groups)
    if identity.idp == 'okta' and not (groups & ALLOWED_GROUPS):
        raise PermissionError(f'DENIED: {identity.email} not in {ALLOWED_GROUPS}')
    if identity.idp == 'azuread' and not groups:
        raise PermissionError(f'DENIED: {identity.email} has no group claim')

def run(idp: str, prompt: str) -> str:
    provider = OktaProvider() if idp == 'okta' else AzureADProvider()
    identity = provider.get_subject_token()
    _audit(f'idp={identity.idp} human={identity.email} subject={identity.subject}')
    authorize(identity)
    _audit(f'authorized human={identity.email} for agent={AGENT_SA}')
    cred = mint_agent_credential(subject_token=identity.id_token, subject=identity.subject,
                                 audience=STS_AUDIENCE, idp=identity.idp,
                                 agent_sa_email=AGENT_SA, lifetime_seconds=TOKEN_TTL)
    _audit(f'minted agent credential sa={cred.agent_sa_email} obo={cred.on_behalf_of_subject} ttl={cred.expires_in}s')
    gemini = GeminiEnterpriseClient(PROJECT_ID, REGION, GEMINI_MODEL)
    answer = gemini.generate(agent_access_token=cred.access_token,
                             on_behalf_of=cred.on_behalf_of_subject, prompt=prompt)
    _audit(f'gemini call complete model={GEMINI_MODEL} agent={cred.agent_sa_email} obo={cred.on_behalf_of_subject}')
    return answer

print('Agent Gateway ready.')

## 5. Demo — Okta front door

A human federated from **Okta** drives the agent. Watch the audit lines: the Gemini call carries the agent SA **and** the human.

In [ ]:
answer = run('okta', 'Summarize our Q3 security posture in three bullets.')
print('\n=== Gemini Enterprise (via Agent Gateway) ===')
print(answer)

## 6. Demo — Microsoft Entra ID front door

Same agent, same gateway — only the federation front door changes.

In [ ]:
answer = run('azuread', 'Draft a 3-step incident response timeline.')
print('\n=== Gemini Enterprise (via Agent Gateway) ===')
print(answer)

## 7. Least-privilege proof — deny at the gateway

The most important thing to show a security reviewer: **without an authorized identity, the model is never reached.**
Here we simulate a user who is *not* in the `gemini-agent-users` group. The request is denied at the policy gate — before any token is minted or any Gemini call is made.

In [ ]:
class UnauthorizedOkta(OktaProvider):
    def get_subject_token(self):
        i = super().get_subject_token()
        i.groups = ['some-other-group']  # not in ALLOWED_GROUPS
        i.email = 'contractor@external.example'
        return i

try:
    ident = UnauthorizedOkta().get_subject_token()
    _audit(f'idp={ident.idp} human={ident.email} subject={ident.subject}')
    authorize(ident)
    print('reached Gemini — THIS SHOULD NOT HAPPEN')
except PermissionError as e:
    print('[DENIED at the gateway]', e)
    print('   No agent token minted. No Gemini call made. Model never reached.')

## 8. Live mode — pointing at a real GCP project

Everything above ran with `DEMO_MODE=1`. To run against a real project:

1. **Provision** the federation pools, agent service account, and Gemini access with the Terraform in the repo:
   ```bash
   git clone https://github.com/avnit/demo-agent-gateway-agent-id.git
   cd demo-agent-gateway-agent-id/terraform
   cp terraform.tfvars.example terraform.tfvars   # fill in project, org, Okta, Entra
   terraform init && terraform apply
   ```
2. **Authenticate** the Colab runtime (so the STS + impersonation hops have a caller):
   ```python
   from google.colab import auth
   auth.authenticate_user()
   ```
3. **Set the real values and flip the switch**, then re-run the gateway cells:
   ```python
   os.environ['DEMO_MODE'] = '0'
   os.environ['GCP_PROJECT_ID'] = 'your-project'
   os.environ['AGENT_SA_EMAIL'] = 'gemini-support-agent@your-project.iam.gserviceaccount.com'
   os.environ['WORKFORCE_POOL'] = 'agent-gateway-workforce'
   os.environ['OKTA_ID_TOKEN'] = '<real Okta OIDC id_token>'   # or AZUREAD_ID_TOKEN
   ```

The caller identity needs `roles/iam.serviceAccountTokenCreator` on the agent SA, and the agent SA needs
`roles/aiplatform.user` on the Gemini model — both are created by the Terraform.

> ⚠️ Live mode makes billable Vertex AI calls. Review IAM bindings and your project before flipping `DEMO_MODE=0`.

---

**Repo:** https://github.com/avnit/demo-agent-gateway-agent-id  •  **License:** Apache-2.0